# Clean and scanned PDFs

Use the same small GLiNER model and parameters as the plain-text baseline. Compare both PDF inputs with strict and relaxed scoring.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'ner').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ner.evaluation import ENTITY_LABELS, evaluate_ner_strict, evaluate_ner_relaxed
from ner.utils import load_jsonl, save_responses_to_jsonl
from ner.inference import load_gliner_model, predict_dataset
from ner.pdf import extract_pdf_dataset
from ner.experiments import evaluate_experiment
gold = load_jsonl(ROOT / 'data/plain/validation.jsonl')

In [ ]:
for input_name in ['pdf_clean', 'pdf_scanned']:
    samples = extract_pdf_dataset(ROOT / 'data' / input_name, gold)
    print(input_name, 'documents:', len(samples), 'empty texts:', sum(not s['text'].strip() for s in samples))
    output = ROOT / f'results/{input_name}/gliner_small.jsonl'
    if output.exists():
        predictions = load_jsonl(output)
    else:
        model = load_gliner_model()
        predictions = predict_dataset(model, samples, list(ENTITY_LABELS), threshold=0.5)
        save_responses_to_jsonl(predictions, output)
        del model
    by_id = {s['id']: s for s in gold}
    aligned = output.with_name('notebook_gold.jsonl')
    save_responses_to_jsonl([by_id[p['id']] for p in predictions], aligned)
    strict = evaluate_ner_strict(aligned, output)
    relaxed = evaluate_ner_relaxed(aligned, output)

PDF extraction can change whitespace and character positions. Strict scores therefore measure both extraction alignment and NER. Relaxed scores ignore offsets and normalize whitespace, but still require an exact mention text and label. The ten scanned files have no extractable text layer; their direct PyMuPDF baseline consequently has zero recall.